---
# Chapter 7 — Why

## Orientation

| Field | Value |
|-------|-------|
| Chapter | 7: Why |
| Central question | Why should a remembered claim be believed? What distinguishes retrieval causality from evidential support? |
| Main concepts | Evidence lineage, Support groups, Derivation lineage, Echo vs corroboration, Raw-source grounding |
| Implementation | evidence_lineage |
| Experiment | Evidence lineage on controlled fixture |
| Evidence status | Book result: core when claims are derived or justified |
| Depends on | Chapter 4 (derived graph), Chapter 2 (instrument) |

---

## What this notebook demonstrates

This chapter supplies one of the strongest durable distinctions in the book:

```text
retrieved because of
≠
supported by
```

The notebook:

1. **Loads the evidence_lineage implementation**
2. **Extracts claims, proposes evidence, builds/inspects support groups/lineage**
3. **Demonstrates the difference** between retrieval causality, derivation lineage, and evidential support
4. **Includes a case** where a topically related source does not actually support the claim

> **Evidence status**: Book result. On the controlled fixture, explicit support structure repaired failures that source pointers and graph connectivity could not.

## The chapter question

> **Why should a remembered claim be believed?**

A query path, graph edge, router choice, or repeated restatement does not license a claim. Evidence lineage added support groups, derivation lineage, echo-versus-corroboration, reverse impact, and raw-source grounding.

## Concepts in this chapter

In [ ]:
import sys
from pathlib import Path

def _find_repo_root(start):
    cur = Path(start).resolve()
    while True:
        if ((cur / "content").is_dir() and (cur / "notebooks").is_dir()
                and (cur / "solution").is_dir()):
            return cur
        if cur == cur.parent:
            raise RuntimeError("could not locate repository root")
        cur = cur.parent

REPO_ROOT = _find_repo_root(Path.cwd())
for _p in (str(REPO_ROOT), str(REPO_ROOT / "solution")):
    if _p not in sys.path:
        sys.path.insert(0, _p)

from notebooks.memory._support import load_chapter_metadata, render_table

meta = load_chapter_metadata(7)
concepts = (meta.get('chapter', {}).get('concepts')
            or meta.get('concepts', []))
render_table([
    {"Concept ID": c['id'], "Name": c['name'], "Status": c['status']}
    for c in concepts
], "Chapter 7 Concepts")

## Load the evidence_lineage implementation

In [ ]:
from evidence_lineage import (
    Claim, extract_claims_c0, extract_claims_c1, extract_claims_c2,
)
from evidence_lineage import fixtures as E7
from evidence_lineage.verification import STAGES, verify_backward

print("Claim fields:", list(Claim.__dataclass_fields__.keys()))
print(f"Fixture: {len(E7.CLAIMS)} ledger claims, {len(E7.SPANS)} source spans")
print("Verification stages:", STAGES)
print("Ledger support groups:", len(E7.SUPPORT_GROUPS))

## The critical distinction: Retrieval causality ≠ Evidential support

| Concept | What it means |
|---------|---------------|
| **Retrieval causality** | Why this item was retrieved (vector similarity, graph traversal, keyword match) |
| **Derivation lineage** | How this derived object was constructed (which text units, which extractor prompt) |
| **Evidential support** | Whether the sources actually *license* the claim (entailment, corroboration, contradiction) |

> **The important architectural result is not a graph type. It is an invariant:** Any derived belief that matters later should remain traceable to the evidence that licences it and to the evidence that would force it to be reconsidered.

## Build a claim and its support group

In [ ]:
# Extract checkable claims from the adr-007 rationale, then show the
# ledger truth about what actually licenses the book's central claim.
spans = {sid: text for sid, _, text in E7.SPANS}
rationale = spans["a07-rationale"]
print("Source span (adr-007):", rationale[:110], "...\n")

for name, fn in [("c0-sentence", extract_claims_c0),
                 ("c1-structured", extract_claims_c1),
                 ("c2-claimify-inspired", extract_claims_c2)]:
    out = fn(rationale, artifact="adr-007")
    live = [c for c in out if not c.abstained]
    print(f"{name}: {len(live)} claims, {len(out) - len(live)} abstentions")
    for c in live:
        print(f"  [{c.id}] {c.text[:75]}")

print("\nLedger support groups for claim-main (OR of ANDs):")
for g in E7.SUPPORT_GROUPS["claim-main"]:
    print("  AND:", sorted(g))

## The case where a topically related source does NOT support the claim

The Redis rejection (`adr-009`) is topically related (caching, performance) but does **not** support the PostgreSQL decision. A system that cites `adr-009` as evidence for PostgreSQL is fabricating support.

In [ ]:
# The provenance-precision trap: topically related, retrievable, and
# licensing nothing. Retrieval causality is not evidential support.
print("Topical-but-non-supporting pairs (claim x span):")
for claim, span in sorted(E7.TOPICAL_ONLY):
    print(f"  {claim} x {span}")
print("\nKey traps:")
print("  claim-main x r06-echo — the runbook restates the decision;")
print("    an echo, not independent corroboration.")
print("  claim-14pct x s33-false — a traceable fabrication; it grounds")
print("    the derivation trace but the claim is still false.")

## Raw-source grounding

The invariant: every derived belief must remain traceable to raw sources. The evidence lineage maps:

1. **Claim** → **support groups** (ledger OR-of-ANDs: what licences it)
2. **Support groups** → **raw source spans** (canonical artifacts, Layer 1)
3. **Derivation lineage** → **derived graph objects** (Layer 2, for audit only)

Raw grounding guarantees inspectability, not truth. But without inspectability, correction has nowhere to begin.

In [ ]:
# The lineage graph plus VeriTrail-style backward verification.
# claim-main holds end to end; the two injected failures localise to
# the exact stage that introduced them, not to the reader.
graph = E7.build_graph()
print(f"Lineage graph: {len(graph.nodes)} nodes, {len(graph.edges)} edges")

for cid in ("claim-main", "claim-14pct", "claim-redis"):
    v = verify_backward(cid, E7.STAGE_SUPPORT[cid])
    print(f"\n{cid}: {v.final_verdict} (error stage: {v.error_stage})")
    for s in v.stages:
        print(f"  {s.stage:12s} supported_by_inputs={s.supported_by_inputs} [{s.basis}]")

## What this establishes

- **Retrieved because of ≠ supported by** — the core invariant
- **Explicit support structure** repaired fixture failures that source pointers and graph connectivity could not
- **Raw grounding guarantees inspectability** — correction starts from canonical sources
- **Echo vs corroboration distinguished** — mere repetition ≠ independent evidence
- **Reverse impact tracked** — what evidence would force reconsideration
- **Core when claims are derived or justified** — not needed for simple lookup

## What this does NOT establish

- No claim that this solves all justification problems
- Real-corpus extraction quality for evidence lineage untested
- Model-judged entailment for corroboration remains pending

## Try it yourself

Create a claim with mixed support/contradiction and see how the lineage evaluates it. Try the temporal claim: "PostgreSQL has always been the choice" — what sources support/contradict?

In [ ]:
# TRY IT YOURSELF: the echo ladder. Five graded restatements of the
# adr-007 decision; a token-overlap detector is measured at each rung
# to map where surface matching stops working.
from evidence_lineage.fixtures import echo_detector

for rung in E7.ECHO_LADDER:
    hit = echo_detector(rung["text"], spans["a07-decision"], 0.5)
    print(f"{rung['id']:14s} echo={hit}  {rung['text'][:65]}")

## Where this leads next

Chapter 8 asks: **Is it still true?** — tracking belief through time with temporal trajectories and supersession.

> **See this chapter in code:** [Open the companion Jupyter notebook](memory-chapter.ipynb)